In [18]:
# Objective:
# Read -> Transform -> Filter -> Write

from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import col
import traceback

# STEP 1 : Create Spark Session
spark = (
    SparkSession.builder
    .appName("Week6_Pipeline")
    .master("local[*]")
    .config(
        "spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version",
        "2"
    )
    .config("spark.sql.sources.commitProtocolClass",
            "org.apache.spark.sql.execution.datasources.SQLHadoopMapReduceCommitProtocol")
    .config("spark.hadoop.fs.file.impl",
            "org.apache.hadoop.fs.LocalFileSystem")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark Session Created Successfully")


Spark Session Created Successfully


In [19]:
# STEP 2 : Define Explicit Schema
schema = StructType([
    StructField("InvoiceNo", StringType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("InvoiceDate", StringType(), True),
    StructField("UnitPrice", DoubleType(), True),
    StructField("CustomerID", IntegerType(), True),
    StructField("Country", StringType(), True)
])

In [20]:
# STEP 3 : Read CSV File
print("\n READING CSV ")

csv_df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("data.csv")

print("CSV Loaded Successfully.")

csv_df.printSchema()

csv_df.show(5, truncate=False)

print("CSV Record Count :", csv_df.count())


 READING CSV 
CSV Loaded Successfully.
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)

+---------+---------+-----------------------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate   |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+--------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |12/1/2010 8:26|2.55     |17850     |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |12/1/2010 8:26|3.39     |17850     |United Kingdom|
|536365   |84406B   |CREAM CUPID

In [21]:
# STEP 4 : Read Parquet File
print("\n READING PARQUET ")

try:

    parquet_df = spark.read.parquet("data.parquet")

    print("Parquet Loaded Successfully.")

    parquet_df.printSchema()

    parquet_df.show(5, truncate=False)

    print("Parquet Record Count :", parquet_df.count())

except Exception:

    print("Unable to read Parquet File.\n")
    traceback.print_exc()


 READING PARQUET 
Parquet Loaded Successfully.
root
 |-- InvoiceNo: integer (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)

+---------+---------+-----------------------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate   |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+--------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |12/1/2010 8:26|2.55     |17850     |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |12/1/2010 8:26|3.39     |17850     |United Kingdom|
|536365   |84406B   |CR

In [22]:
# STEP 5 : Handle Null Values
print("\n HANDLING NULL VALUES ")

df = csv_df.fillna({
    "Description": "Unknown Product",
    "CustomerID": 0
})



 HANDLING NULL VALUES 


In [23]:
# STEP 6 : Rename Column
print("\n RENAMING COLUMN ")

df = df.withColumnRenamed(
    "Description",
    "ProductDescription"
)



 RENAMING COLUMN 


In [24]:
# STEP 7 : Cast Data Types

print("\n CASTING DATA TYPES ")

df = df.withColumn("Quantity", col("Quantity").cast("int"))

df = df.withColumn("UnitPrice", col("UnitPrice").cast("double"))



 CASTING DATA TYPES 


In [25]:
# STEP 8 : Add New Column
print("\n ADDING NEW COLUMN ")

df = df.withColumn(
    "TotalAmount",
    col("Quantity") * col("UnitPrice")
)



 ADDING NEW COLUMN 


In [26]:
# STEP 9 : Filter Records
print("\n FILTERING DATA ")

filtered_df = df.filter(
    (col("Quantity") > 0) &
    (col("UnitPrice") > 0)
)



 FILTERING DATA 


In [27]:
# STEP 10 : Select Required Columns
print("\n SELECTING REQUIRED COLUMNS ")

final_df = filtered_df.select(
    "InvoiceNo",
    "StockCode",
    "ProductDescription",
    "Quantity",
    "UnitPrice",
    "TotalAmount",
    "Country"
)



 SELECTING REQUIRED COLUMNS 


In [28]:
# STEP 11 : Display Processed Data
print("\n PROCESSED DATA ")

final_df.show(10, truncate=False)

print("Processed Record Count :", final_df.count())


 PROCESSED DATA 
+---------+---------+-----------------------------------+--------+---------+------------------+--------------+
|InvoiceNo|StockCode|ProductDescription                 |Quantity|UnitPrice|TotalAmount       |Country       |
+---------+---------+-----------------------------------+--------+---------+------------------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |2.55     |15.299999999999999|United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |3.39     |20.34             |United Kingdom|
|536365   |84406B   |CREAM CUPID HEARTS COAT HANGER     |8       |2.75     |22.0              |United Kingdom|
|536365   |84029G   |KNITTED UNION FLAG HOT WATER BOTTLE|6       |3.39     |20.34             |United Kingdom|
|536365   |84029E   |RED WOOLLY HOTTIE WHITE HEART.     |6       |3.39     |20.34             |United Kingdom|
|536365   |22752    |SET 7 BABUSHKA NESTING BOXES       |2       |7.65     |15.3              

In [29]:
# STEP 12 : Write Processed CSV
print("\n WRITING CSV ")

try:

    (final_df
 .coalesce(1)
 .write
 .mode("overwrite")
 .option("header", "true")
 .csv(r"D:\CELEBAL\WEEK 6\processed_csv"))

    print("Processed CSV Saved Successfully.")

except Exception:

    print("\nCSV Writing Failed.\n")
    traceback.print_exc()


 WRITING CSV 
Processed CSV Saved Successfully.


In [30]:
# STEP 13 : Write Processed Parquet
print("\n WRITING PARQUET n")

try:

    (final_df
 .coalesce(1)
 .write
 .mode("overwrite")
 .parquet(r"D:\CELEBAL\WEEK 6\processed_parquet"))

    print("Processed Parquet Saved Successfully.")

except Exception:

    print("\nParquet Writing Failed.\n")
    traceback.print_exc()



 WRITING PARQUET n
Processed Parquet Saved Successfully.


In [31]:
# STEP 14 : Performance Insights
print("\n PERFORMANCE INSIGHTS ")

print("1. Spark uses Lazy Evaluation.")
print("2. Transformations execute only when an Action is called.")
print("3. show() and count() are Actions.")
print("4. filter(), select(), withColumn() are Transformations.")
print("5. Parquet is faster than CSV because it is columnar.")
print("6. Spark creates a DAG before execution.")
print("7. Predicate Pushdown improves Parquet performance.")
print("8. Avoid collect() on large datasets.")


 PERFORMANCE INSIGHTS 
1. Spark uses Lazy Evaluation.
2. Transformations execute only when an Action is called.
3. show() and count() are Actions.
4. filter(), select(), withColumn() are Transformations.
5. Parquet is faster than CSV because it is columnar.
6. Spark creates a DAG before execution.
7. Predicate Pushdown improves Parquet performance.
8. Avoid collect() on large datasets.


In [32]:
# STEP 15 : Stop Spark
spark.stop()
print("Spark Session Stopped Successfully.")
print("Program Executed Successfully.")

Spark Session Stopped Successfully.
Program Executed Successfully.
